# V42 — Matched DID event study with sugar-based yogurt and cereal

Save this notebook in `analysis/event_study/` and run cells in order.
Input: `hmm_estimation_panel_sugar.parquet`. The restricted Parquet must be
available locally; this notebook does not edit it.

**Primary outcome:** household-year sum of healthy-alternative spending divided
by household-year sum of trip `food_spend`. The **five research categories** are
FV, cereal, yogurt, snacks, and combined beverages.

| Category | Spending included in the healthy numerator |
| --- | --- |
| FV | `fresh_spending` |
| Cereal | `low_sugar_cereal_spending` + `medium_sugar_cereal_spending` |
| Yogurt | `regular_sugar_spending` |
| Snacks | `healthy_snack_spending` |
| Combined beverages | `healthy_beverage_spending` + `skimmed_milk_spending` + `low_fat_milk_spending` + `other_milk_spending` |

Regular-sugar yogurt is the healthier yogurt alternative; `high_sugar_spending`
is excluded. This replaces the former fat-based yogurt definition. Both low-
and medium-sugar cereal are healthy; high-sugar cereal spending is excluded.
Milk at every fat level remains in healthy beverages and is not a separate
category. Each source spending column is counted exactly once.

**Spending and choice are distinct:** source `yogurt_y` codes are 0 = no
classified purchase, 1 = regular sugar, 2 = high sugar. Source `cereal_y` codes
are 0 = no classified purchase, 1 = low sugar, 2 = medium sugar without low,
3 = high sugar without low or medium. These codes audit category purchase;
they do not allocate all trip spending to a single alternative. A mixed basket
contributes only its included healthier-alternative dollars to the numerator.

**Design retained from the previous matched DID:** separate analyses for six
life events; first *observed* occurrence, no repeat of the focal event within
[-3,+3], and a complete seven-year outcome window. Controls have no observed
occurrence of the focal event anywhere in the input panel and a complete
window in the same calendar years. Other life events may occur in either
group. With 2008–2017 data, full-window event cohorts are 2011–2014.

Matching within cohort uses 1:1 propensity-score nearest-neighbor matching
with replacement, exact region measured at -1, and a 0.2-SD logit caliper.
Propensity models use Y at -3, -2, -1 and baseline age, income, household
size, and child presence. The outcome changed, so **matching is rerun**.

Event-time DID is the matched treated-minus-control change relative to year -1,
in percentage points. Inference retains the original household-clustered
fixed-match sandwich and pointwise 90% intervals. Propensity estimation and
matching uncertainty are not included. Pretrend tests are partly constrained
by matching on pre-event outcomes; complete windows select on panel retention.


**Added spending fit comparison:** Section 6 compares observed and model-implied
healthy spending shares using exactly the same outcome as the main spending DID.
Actual trip-category budgets are held fixed. State-independent auxiliary amount
fractions are estimated with household cross-fitting and combined with the fitted
V42 conditional-choice probabilities and smoothed posteriors. Both curves reuse
the main spending-DID matched pairs, cohorts, control reuse and weights.
Run the supplied V42 HMM first and set `MODEL_RESULTS_DIR` to its completed
sample folder. The HMM is read from exports and is not re-estimated.


In [ ]:
# Cell 1 is deliberately standard-library-only: it does not import scientific
# libraries, scan parquet data, build font caches, or install packages.
print("[START] Python kernel is executing this cell.", flush=True)
from pathlib import Path
import os
import sys
import time
import json
import warnings
import importlib
import importlib.util
import importlib.metadata
import threading
from contextlib import contextmanager

# Set before importing NumPy/SciPy; restart the kernel for these to take effect.
for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[variable] = "1"
print(f"Python: {sys.version.split()[0]} | Executable: {sys.executable}", flush=True)

@contextmanager
def progress(label):
    """Show elapsed time, not a fabricated completion percentage."""
    started = time.monotonic()
    done = threading.Event()
    print(f"[START] {label}", flush=True)
    def heartbeat():
        while not done.wait(15):
            print(f"[RUNNING] {label}: {time.monotonic()-started:.0f}s elapsed", flush=True)
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    try:
        yield
    except BaseException:
        print(f"[STOPPED] {label}; see traceback below.", flush=True)
        raise
    else:
        print(f"[DONE] {label}: {time.monotonic()-started:.1f}s", flush=True)
    finally:
        done.set()
        worker.join(timeout=0.2)

def load_module(name):
    if name in sys.modules:
        return sys.modules[name]
    with progress(f"Import {name}"):
        return importlib.import_module(name)

DATA_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel_sugar.parquet")
OUTPUT_DIR = Path(r"C:\Users\rwen13\Documents\healthy_purchase_cycle\model_free_evidence\v42_sugar_yogurt_low_medium_cereal")
PRIVATE_DIR = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\model_free_evidence\v42_sugar_yogurt_low_medium_cereal")

# Run the supplied V42 HMM first. Use its sample-specific result directory.
# This default matches the attached fast model's settings (2008–2017, minHH3).
MODEL_RESULTS_DIR = (DATA_PATH.parent / "V42_Sugar_Choice_Bootstrap_Results_Optimized"
                     / "years_2008_2017_minHH3_observed_step")
RUN_HEALTHY_SPENDING_COMPARISON = True
# Auxiliary amount allocation only: the fitted HMM is never re-estimated.
AMOUNT_AUX_FOLDS = 5          # Hold out complete household histories.
AMOUNT_AUX_SEED = 20261118
AMOUNT_AUX_RIDGE = 1e-4      # Prediction regularization; intercept is unpenalized.
AMOUNT_BUDGET_CUTS = [2.0, 5.0, 10.0, 20.0]  # Observed per-trip category dollars.
USE_AMOUNT_CALIBRATION_CACHE = True

INCLUDE_ALL_EXISTING_MILK_IN_BEVERAGE = True
YEAR_START, YEAR_END = 2008, 2017
PRE, POST, REFERENCE = 3, 3, -1
CONFIDENCE_LEVEL = 0.90  # Two-sided intervals; significance level = 10%.
CALIPER_SD = 0.20
EXACT_REGION = True
MIN_TREATED, MIN_CONTROLS = 20, 20  # Eligibility thresholds before matching.
MIN_MATCHED = 20                   # Minimum retained pairs per cohort.
BALANCE_THRESHOLD = 0.10          # Diagnostic flag, not automatic deletion.
CHECK_DUPLICATE_TRIPS = True
USE_ANNUAL_CACHE = True
SAVE_PRIVATE_MATCHES = True
DUCKDB_MEMORY_LIMIT = "2GB"
DUCKDB_THREADS = min(4, max(1, (os.cpu_count() or 2) // 2))
AGGREGATION_MODE = "auto"  # auto, single_pass, or yearwise
RANDOM_SEED = 20261031
PIPELINE_VERSION = "V42-sugar-yogurt-low-medium-cereal-spending-v1"

EVENT_LABELS = {
    "event_new_child": "New child",
    "getting_married": "Marriage",
    "getting_divorce_or_separation": "Divorce / separation",
    "moving": "Moving",
    "getting_job": "Starting a job",
    "retirement": "Retirement",
}
# Original source columns: no derived HMM outcome is written to the Parquet.
# All three mutually exclusive ordinary-milk spending columns enter the
# healthy BEVERAGE numerator, irrespective of milk fat content.
# Each outcome maps to its mutually exclusive healthier spending components.
HEALTHY_COLUMNS = {
    "fv_y": ["fresh_spending"],
    "cereal_y": ["low_sugar_cereal_spending", "medium_sugar_cereal_spending"],
    "yogurt_y": ["regular_sugar_spending"],
    "snack_y": ["healthy_snack_spending"],
    "beverage_y": ["healthy_beverage_spending"],
}
OUTCOME_ALLOWED_CODES = {
    "fv_y": [0, 1, 2], "cereal_y": [0, 1, 2, 3],
    "yogurt_y": [0, 1, 2], "snack_y": [0, 1, 2],
    "beverage_y": [0, 1, 2], "milk_y": [0, 1, 2],
}
MILK_SPENDING_COLUMNS = [
    "skimmed_milk_spending",
    "low_fat_milk_spending",
    "other_milk_spending",
]
HEALTHY_SPENDING_COLUMNS = [spend for columns in HEALTHY_COLUMNS.values() for spend in columns] + MILK_SPENDING_COLUMNS
assert len(HEALTHY_SPENDING_COLUMNS) == len(set(HEALTHY_SPENDING_COLUMNS))
RESEARCH_CATEGORIES = ["fv", "cereal", "yogurt", "snack", "beverage_combined"]

COVARIATES = ["age", "income10k_imputed", "family_size_numeric", "K_1"]
ANNUAL_COVARIATES = COVARIATES + (["regional_cd"] if EXACT_REGION else [])
EVENT_TIMES = list(range(-PRE, POST + 1))
assert PRE >= 2 and REFERENCE == -1 and POST >= 0

print("[READY] Settings loaded. No purchase data have been read.", flush=True)


## Startup instructions (repaired version)
**Restart the kernel first**, then run cells in order. Cell 1 prints START
immediately and uses only Python's standard library. If that line never
appears, the selected Jupyter kernel has not started executing Python; restart
or reselect the Python kernel in VS Code before running further cells.
Cell 2 lists missing dependencies and logs each import. It never installs or
upgrades packages automatically. If an import stalls, the last START message
identifies the module. First Matplotlib import is deferred until plotting,
with a writable local temporary font-cache directory.
Each scan/aggregation stage prints START/DONE plus an elapsed
time message every 15 seconds. Elapsed messages are activity indicators,
not proof that a slow native operation is making progress.


In [ ]:
packages = {"numpy": "numpy", "pandas": "pandas", "duckdb": "duckdb",
            "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib"}
missing = [pip_name for module, pip_name in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Run in a separate notebook cell, then restart the kernel:", flush=True)
    print("%pip install " + " ".join(missing), flush=True)
    raise ImportError("Missing packages: " + ", ".join(missing))
np = load_module("numpy")
pd = load_module("pandas")
duckdb = load_module("duckdb")
load_module("pyarrow")
from IPython.display import display
EVENT_TIMES = np.array(EVENT_TIMES, dtype=int)
print("[READY] Data libraries imported. Next cells define functions only.", flush=True)


## 1. Validate inputs and aggregate the sugar-panel spending outcome

The input is `hmm_estimation_panel_sugar.parquet`. Healthy yogurt spending is
`regular_sugar_spending`; the old fat-based yogurt columns are not used.
Healthy cereal spending adds the low- and medium-sugar components. High-sugar
yogurt and cereal do not enter the healthy numerator. All three ordinary-milk
spending columns enter healthy combined beverages exactly once.

Source outcome codes are validated before aggregation: cereal permits
{0,1,2,3}; the other five source categories permit {0,1,2}. Missing or invalid
codes fail the audit. Missing included spending is zero-filled only for
category nonpurchase (code 0). Missing amounts on category-purchase trips,
negative/nonfinite spending, invalid events, varying annual demographics and
duplicate trip keys fail the audit. No clipping or post hoc adjustment of
annual shares is performed.

The V42 cache includes the source file, spending definitions, allowed codes,
and pipeline version. Previous fat-yogurt or low-only cereal caches are not
reused. The outcome change also requires new propensity models and matches.


In [ ]:
def qi(name):
    return '"' + name.replace('"', '""') + '"'


def build_annual_panel(data_path, private_dir, table_dir):
    data_path, private_dir, table_dir = map(Path, (data_path, private_dir, table_dir))
    if not data_path.is_file():
        raise FileNotFoundError(f"Input parquet not found: {data_path}")
    private_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)
    stamp = {
        "pipeline_version": PIPELINE_VERSION,
        "source": str(data_path.resolve()),
        "size": data_path.stat().st_size,
        "mtime_ns": data_path.stat().st_mtime_ns,
        "healthy_columns": HEALTHY_COLUMNS,
        "milk_spending_columns": MILK_SPENDING_COLUMNS,
        "healthy_spending_columns": HEALTHY_SPENDING_COLUMNS,
        "outcome_allowed_codes": OUTCOME_ALLOWED_CODES,
        "combined_beverage_rule": "all-existing-ordinary-milk-fat-levels",
        "annual_covariates": ANNUAL_COVARIATES,
        "years": [YEAR_START, YEAR_END],
        "duplicate_check": CHECK_DUPLICATE_TRIPS,
    }
    cache = private_dir / "annual_v42_sugar_yogurt_low_medium_cereal.parquet"
    meta = cache.with_suffix(".json")
    audit_cache = cache.with_suffix(".audit.csv")
    if USE_ANNUAL_CACHE and cache.exists() and meta.exists() and audit_cache.exists():
        if json.loads(meta.read_text(encoding="utf-8")) == stamp:
            panel = pd.read_parquet(cache)
            audit = pd.read_csv(audit_cache)
            audit.to_csv(table_dir / "input_audit.csv", index=False)
            print(f"Loaded validated annual cache: {len(panel):,} household-years")
            return panel, audit

    con = duckdb.connect()
    try:
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        con.execute(f"SET threads={int(DUCKDB_THREADS)}")
        temp_sql = (private_dir / "duckdb_temp").as_posix().replace("'", "''")
        con.execute(f"SET temp_directory='{temp_sql}'")
        source_sql = data_path.as_posix().replace("'", "''")
        con.execute(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{source_sql}')")
        names = set(con.execute("DESCRIBE trips").fetchdf()["column_name"])
        required = ({"hhid", "panel_year", "trip_id", "purchase_date", "food_spend"}
                    | set(EVENT_LABELS) | set(ANNUAL_COVARIATES)
                    | set(HEALTHY_COLUMNS) | set(HEALTHY_SPENDING_COLUMNS)
                    | {"milk_y", "beverage_y"})
        if required - names:
            raise ValueError(f"Missing required columns: {sorted(required - names)}")

        checks = {
            "trip_rows": "COUNT(*)",
            "missing_key_rows": "COUNT(*) FILTER (WHERE hhid IS NULL OR panel_year IS NULL OR trip_id IS NULL OR purchase_date IS NULL)",
            "outside_year_rows": f"COUNT(*) FILTER (WHERE panel_year NOT BETWEEN {YEAR_START} AND {YEAR_END})",
            "date_year_mismatch_rows": "COUNT(*) FILTER (WHERE YEAR(purchase_date) <> panel_year)",
            "invalid_food_rows": "COUNT(*) FILTER (WHERE food_spend IS NULL OR NOT isfinite(food_spend) OR food_spend < 0)",
            "zero_food_trip_rows": "COUNT(*) FILTER (WHERE food_spend = 0)",
        }
        # Validate source codes; null is invalid, not equivalent to nonpurchase.
        for outcome, allowed in OUTCOME_ALLOWED_CODES.items():
            permitted_sql = ",".join(map(str, allowed))
            checks[f"invalid_code__{outcome}"] = (
                f"COUNT(*) FILTER (WHERE {qi(outcome)} IS NULL "
                f"OR NOT isfinite(CAST({qi(outcome)} AS DOUBLE)) "
                f"OR {qi(outcome)} NOT IN ({permitted_sql}))"
            )
        for outcome, spend in ((outcome, spend)
                               for outcome, columns in HEALTHY_COLUMNS.items()
                               for spend in columns):
            checks[f"invalid__{spend}"] = f"COUNT(*) FILTER (WHERE ({qi(spend)} IS NULL AND {qi(outcome)} NOT IN (0)) OR ({qi(spend)} IS NOT NULL AND (NOT isfinite({qi(spend)}) OR {qi(spend)} < 0)))"
            checks[f"nonpurchase_null__{spend}"] = f"COUNT(*) FILTER (WHERE {qi(spend)} IS NULL AND ({qi(outcome)} IS NULL OR {qi(outcome)} = 0))"
        for spend in MILK_SPENDING_COLUMNS:
            checks[f"invalid__{spend}"] = (
                f"COUNT(*) FILTER (WHERE ({qi(spend)} IS NULL AND milk_y IN (1,2)) "
                f"OR ({qi(spend)} IS NOT NULL AND "
                f"(NOT isfinite({qi(spend)}) OR {qi(spend)} < 0)))"
            )
            checks[f"nonpurchase_null__{spend}"] = (
                f"COUNT(*) FILTER (WHERE {qi(spend)} IS NULL AND milk_y = 0)"
            )
        for event in EVENT_LABELS:
            checks[f"invalid__{event}"] = f"COUNT(*) FILTER (WHERE {qi(event)} IS NULL OR {qi(event)} NOT IN (0,1))"
        expressions = [f"{expr} AS {qi(name)}" for name, expr in checks.items()]
        with progress("Audit trip amounts, keys and event codes"):
            raw = con.execute("SELECT " + ", ".join(expressions) + " FROM trips").fetchdf().iloc[0]
        audit_rows = [{"metric": k, "value": int(v)} for k, v in raw.items()]
        fatal = [k for k in raw.index if (k.startswith("invalid") or k in
                 ["missing_key_rows", "outside_year_rows", "date_year_mismatch_rows"]) and raw[k] > 0]
        if raw["trip_rows"] == 0:
            fatal.append("empty_input")
        if fatal:
            pd.DataFrame(audit_rows).to_csv(table_dir / "input_audit.csv", index=False)
            raise ValueError(f"Input audit failed; see input_audit.csv: {fatal}")

        healthy_sum = " + ".join(f"COALESCE({qi(c)},0)" for c in HEALTHY_SPENDING_COLUMNS)
        aggregates = ["COUNT(*) AS n_trips", "SUM(food_spend) AS food_spend",
                      f"SUM({healthy_sum}) AS healthy_spend"]
        if CHECK_DUPLICATE_TRIPS:
            aggregates.append("COUNT(DISTINCT (trip_id, purchase_date)) AS n_unique_trips")
        for c in list(EVENT_LABELS) + ANNUAL_COVARIATES:
            aggregates += [f"MIN({qi(c)}) AS {qi(c)}",
                           f"MAX({qi(c)}) AS {qi(c + '__max')}",
                           f"COUNT({qi(c)}) AS {qi(c + '__count')}"]
        # Inspect parquet metadata only. Broad year ranges in row groups imply
        # that filtering each year would repeatedly read much of the same file.
        mode = AGGREGATION_MODE
        if mode not in {"auto", "single_pass", "yearwise"}:
            raise ValueError("AGGREGATION_MODE must be auto, single_pass, or yearwise")
        if mode == "auto":
            pq = load_module("pyarrow.parquet")
            with pq.ParquetFile(data_path) as pf:
                col_idx = pf.schema_arrow.get_field_index("panel_year")
                spans = []
                for r in range(pf.metadata.num_row_groups):
                    stat = pf.metadata.row_group(r).column(col_idx).statistics
                    if stat is not None and stat.has_min_max:
                        spans.append(int(stat.max) - int(stat.min) + 1)
                mode = "yearwise" if spans and np.mean(spans) <= 2 else "single_pass"
        print(f"Aggregation mode: {mode}; DuckDB threads: {DUCKDB_THREADS}; memory cap: {DUCKDB_MEMORY_LIMIT}", flush=True)
        if mode == "single_pass":
            try:
                with progress("Aggregate all years in one scan, including duplicate-trip check"):
                    panel = con.execute("SELECT hhid, panel_year, " + ", ".join(aggregates)
                                        + " FROM trips GROUP BY hhid, panel_year").fetchdf()
            except duckdb.OutOfMemoryException:
                print("Memory cap reached; retrying year by year with the same checks.", flush=True)
                mode = "yearwise"
        if mode == "yearwise":
            yearly = []
            for year in range(YEAR_START, YEAR_END + 1):
                with progress(f"Aggregate year {year} (including duplicate-trip check)"):
                    part = con.execute("SELECT hhid, panel_year, " + ", ".join(aggregates)
                                       + f" FROM trips WHERE panel_year = {year}"
                                       + " GROUP BY hhid, panel_year").fetchdf()
                yearly.append(part)
                print(f"  {year}: {len(part):,} household-years", flush=True)
            panel = pd.concat(yearly, ignore_index=True)
    finally:
        con.close()

    fatal = []
    if CHECK_DUPLICATE_TRIPS:
        duplicates = int((panel.n_trips - panel.n_unique_trips).sum())
        audit_rows.append({"metric": "duplicate_trip_rows", "value": duplicates})
        if duplicates:
            fatal.append("duplicate_trip_rows")
        panel = panel.drop(columns="n_unique_trips")
    for c in list(EVENT_LABELS) + ANNUAL_COVARIATES:
        varying = int((panel[c].notna() & panel[c + "__max"].notna()
                       & (panel[c] != panel[c + "__max"])).sum())
        partial = int(((panel[c + "__count"] > 0)
                       & (panel[c + "__count"] < panel.n_trips)).sum())
        audit_rows += [{"metric": f"varying_within_year__{c}", "value": varying},
                       {"metric": f"partial_missing_within_year__{c}", "value": partial}]
        if varying or partial:
            fatal.append(c)
        panel = panel.drop(columns=[c + "__max", c + "__count"])
    panel["Y"] = panel.healthy_spend.div(panel.food_spend.where(panel.food_spend > 0))
    invalid_share = int(((panel.Y < -1e-10) | (panel.Y > 1 + 1e-10)).sum())
    audit_rows += [
        {"metric": "household_years", "value": len(panel)},
        {"metric": "households", "value": panel.hhid.nunique()},
        {"metric": "nonpositive_annual_food", "value": int((panel.food_spend <= 0).sum())},
        {"metric": "annual_share_outside_0_1", "value": invalid_share},
    ]
    if invalid_share:
        fatal.append("annual_share_outside_0_1")
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(table_dir / "input_audit.csv", index=False)
    if fatal:
        raise ValueError(f"Annual audit failed; see input_audit.csv: {fatal}")
    panel = panel.sort_values(["hhid", "panel_year"]).reset_index(drop=True)
    panel.to_parquet(cache, index=False)
    audit.to_csv(audit_cache, index=False)
    meta.write_text(json.dumps(stamp, indent=2), encoding="utf-8")
    print(f"Aggregated {int(raw['trip_rows']):,} trips into {len(panel):,} household-years")
    return panel, audit


## 2. Match before estimating outcome changes
Exact region uses the **year -1** value, so a move in year 0 is not matched away.
A separate propensity model is fitted inside each eligible calendar cohort.
Scores are lightly regularized logistic regressions (C=1000), without class
reweighting. Common-support trimming uses the overlap of group score ranges.
The caliper is 0.2 times the pooled within-group SD of the original eligible
logit scores, held fixed before trimming. All matching rules are configurable.
Missing baseline features are excluded, not filled with post-event information.
Ties resolve deterministically by sorted control household ID.

The balance table reports cohort-specific standardized mean differences (SMDs)
before and after matching, using the SAME pre-match pooled SD denominator.
Controls repeated in matches receive their reuse frequency as weight.
Balance flags do not trigger hidden rematching or outcome-based model selection.


In [ ]:
def nearest_logit_matches(treated_scores, control_scores):
    """O(n log n) nearest neighbor search; controls must already be ID-sorted."""
    order = np.argsort(control_scores, kind="stable")
    sorted_scores = control_scores[order]
    # Keep the first ID among exact-score ties.
    unique_scores, starts = np.unique(sorted_scores, return_index=True)
    representatives = order[starts]
    right = np.searchsorted(unique_scores, treated_scores)
    a = np.clip(right - 1, 0, len(unique_scores) - 1)
    b = np.clip(right, 0, len(unique_scores) - 1)
    da, db = np.abs(treated_scores - unique_scores[a]), np.abs(treated_scores - unique_scores[b])
    chosen = np.where(da <= db, a, b)
    return representatives[chosen], np.minimum(da, db)


def match_event(panel, event, ywide):
    # Cached Python imports: heavy ML libraries load only when matching starts.
    StandardScaler = load_module("sklearn.preprocessing").StandardScaler
    LogisticRegression = load_module("sklearn.linear_model").LogisticRegression
    ConvergenceWarning = load_module("sklearn.exceptions").ConvergenceWarning
    ever = panel.groupby("hhid")[event].max().reindex(ywide.index)
    first = panel.loc[panel[event].eq(1)].groupby("hhid").panel_year.min()
    first = first.reindex(ywide.index)
    ewide = panel.pivot(index="hhid", columns="panel_year", values=event).reindex(ywide.index)
    pairs, diagnostics, balances = [], [], []
    for cohort in range(YEAR_START + PRE, YEAR_END - POST + 1):
        print(f"  {event}: cohort {cohort}", flush=True)
        window_years = cohort + EVENT_TIMES
        ywindow = ywide.reindex(columns=window_years)
        complete = np.isfinite(ywindow.to_numpy()).all(axis=1)
        ew = ewide.reindex(columns=window_years)
        treated_rule = first.eq(cohort) & ew.sum(axis=1).eq(1)
        control_rule = ever.eq(0)
        baseline = panel.loc[panel.panel_year.eq(cohort - 1)].set_index("hhid")
        features = pd.DataFrame(index=ywide.index)
        for k in range(-PRE, 0):
            features[f"Y_pre{abs(k)}"] = ywide.reindex(columns=[cohort+k]).iloc[:, 0]
        for c in ANNUAL_COVARIATES:
            features[c] = baseline[c].reindex(features.index)
        finite_x = np.isfinite(features.to_numpy(dtype=float)).all(axis=1)
        eligible = complete & finite_x
        t_ids = features.index[eligible & treated_rule.to_numpy()]
        c_ids = features.index[eligible & control_rule.to_numpy()]
        diag = {
            "event": event, "cohort": cohort,
            "first_observed_events": int(first.eq(cohort).sum()),
            "treated_no_repeat_complete_window": int((treated_rule.to_numpy() & complete).sum()),
            "eligible_treated": len(t_ids), "eligible_controls": len(c_ids),
            "matched_pairs": 0, "retained_pairs": 0,
            "status": "insufficient_eligible_sample",
        }
        if len(t_ids) < MIN_TREATED or len(c_ids) < MIN_CONTROLS:
            diagnostics.append(diag)
            continue
        ids = t_ids.append(c_ids)
        fit_features = features.loc[ids].copy()
        # Region enters the score as indicators and is also matched exactly.
        if EXACT_REGION:
            fit_features = pd.get_dummies(fit_features, columns=["regional_cd"], dtype=float)
        x = StandardScaler().fit_transform(fit_features)
        treatment = np.r_[np.ones(len(t_ids)), np.zeros(len(c_ids))]
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            fit = LogisticRegression(C=1000.0, solver="lbfgs", max_iter=2000,
                                     random_state=RANDOM_SEED).fit(x, treatment)
        score = pd.Series(fit.decision_function(x), index=ids)
        ts, cs = score.loc[t_ids], score.loc[c_ids]
        pooled_sd = np.sqrt((ts.var(ddof=1) + cs.var(ddof=1)) / 2)
        caliper = CALIPER_SD * pooled_sd
        lower, upper = max(ts.min(), cs.min()), min(ts.max(), cs.max())
        support = score.between(lower, upper)
        diag.update({"logit_caliper": caliper,
                     "treated_in_common_support": int(support.loc[t_ids].sum()),
                     "controls_in_common_support": int(support.loc[c_ids].sum())})
        t_support = t_ids[support.loc[t_ids].to_numpy()]
        c_support = c_ids[support.loc[c_ids].to_numpy()]
        cohort_pairs = []
        regions = sorted(features.loc[t_support, "regional_cd"].unique()) if EXACT_REGION else [None]
        for region in regions:
            ti, ci = t_support, c_support
            if EXACT_REGION:
                ti = ti[features.loc[ti, "regional_cd"].eq(region).to_numpy()]
                ci = ci[features.loc[ci, "regional_cd"].eq(region).to_numpy()]
            ci = ci.sort_values()
            if len(ti) == 0 or len(ci) == 0:
                continue
            nearest, distance = nearest_logit_matches(score.loc[ti].to_numpy(), score.loc[ci].to_numpy())
            keep = distance <= caliper + 1e-12
            cohort_pairs.append(pd.DataFrame({
                "event": event, "cohort": cohort,
                "treated_hhid": ti[keep], "control_hhid": ci[nearest[keep]],
                "logit_distance": distance[keep],
            }))
        matched = pd.concat(cohort_pairs, ignore_index=True) if cohort_pairs else pd.DataFrame()
        diag["matched_pairs"] = len(matched)
        if len(matched) < MIN_MATCHED:
            diag["status"] = "insufficient_matches_after_caliper"
            diagnostics.append(diag)
            continue
        diag.update({"status": "retained", "retained_pairs": len(matched),
                     "unique_controls": matched.control_hhid.nunique(),
                     "match_rate": len(matched) / len(t_ids)})
        diagnostics.append(diag)
        pairs.append(matched)
        pre_t = fit_features.loc[t_ids].to_numpy(dtype=float)
        pre_c = fit_features.loc[c_ids].to_numpy(dtype=float)
        post_t = fit_features.loc[matched.treated_hhid].to_numpy(dtype=float)
        post_c = fit_features.loc[matched.control_hhid].to_numpy(dtype=float)
        sd = np.sqrt((pre_t.var(axis=0, ddof=1) + pre_c.var(axis=0, ddof=1)) / 2)
        for j, name in enumerate(fit_features.columns):
            before, after = pre_t[:, j].mean() - pre_c[:, j].mean(), post_t[:, j].mean() - post_c[:, j].mean()
            smd_before = before / sd[j] if sd[j] > 0 else (0.0 if abs(before) < 1e-12 else np.nan)
            smd_after = after / sd[j] if sd[j] > 0 else (0.0 if abs(after) < 1e-12 else np.nan)
            balances.append({"event": event, "cohort": cohort, "feature": name,
                             "smd_before": smd_before, "smd_after": smd_after,
                             "absolute_smd_after": abs(smd_after),
                             "balance_flag": not np.isfinite(smd_after) or abs(smd_after) > BALANCE_THRESHOLD})
    pair_table = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()
    return pair_table, pd.DataFrame(diagnostics), pd.DataFrame(balances)


## 3. Estimate matched contrasts and household-clustered covariance
Let n_g be retained pairs in cohort g, N=sum(n_g), and dY=Y_k-Y_-1.
The estimate is sum_g (n_g/N)(mean_T,g(dY)-mean_C,g(dY)), where repeated
control appearances count repeatedly in the weighted control mean.
For each original household, sum its signed, cohort-mean-centered dY residuals
across ALL its matched appearances, divide by N, and take the cross-product
of these household influence vectors. Multiply by G/(G-1), where G is the
number of unique original households. This is the fixed-design clustered
sandwich for the cohort-specific weighted mean contrasts; cohort weights
and matches are held fixed. The covariance includes cross-event-time terms.
Reference -1 has zero estimate/SE by normalization and no p-value.


In [ ]:
def estimate_event(pair_table, ywide, event):
    stats = load_module("scipy.stats")
    n_total = len(pair_table)
    households = pd.Index(sorted(set(pair_table.treated_hhid) | set(pair_table.control_hhid)))
    influence = np.zeros((len(households), len(EVENT_TIMES)))
    estimate = np.zeros(len(EVENT_TIMES))
    cohort_rows = []
    ref_index = int(np.flatnonzero(EVENT_TIMES == REFERENCE)[0])
    for cohort, pairs in pair_table.groupby("cohort", sort=True):
        years = cohort + EVENT_TIMES
        yt = ywide.loc[pairs.treated_hhid, years].to_numpy(dtype=float)
        yc = ywide.loc[pairs.control_hhid, years].to_numpy(dtype=float)
        dt = yt - yt[:, [ref_index]]
        dc = yc - yc[:, [ref_index]]
        mt, mc = dt.mean(axis=0), dc.mean(axis=0)
        contrast = mt - mc
        estimate += len(pairs) / n_total * contrast
        np.add.at(influence, households.get_indexer(pairs.treated_hhid), (dt - mt) / n_total)
        np.add.at(influence, households.get_indexer(pairs.control_hhid), -(dc - mc) / n_total)
        cohort_rows.extend({"event": event, "cohort": int(cohort), "event_time": int(k),
                            "did_pp": 100 * contrast[j], "matched_treated": len(pairs),
                            "cohort_weight": len(pairs) / n_total}
                           for j, k in enumerate(EVENT_TIMES))
    g = len(households)
    covariance = influence.T @ influence * g / (g - 1)
    se = np.sqrt(np.maximum(np.diag(covariance), 0))
    critical = stats.t.ppf((1 + CONFIDENCE_LEVEL) / 2, g - 1)
    pvalues = np.full(len(EVENT_TIMES), np.nan)
    nonzero = se > 0
    pvalues[nonzero] = 2 * stats.t.sf(np.abs(estimate[nonzero] / se[nonzero]), g - 1)
    rows = pd.DataFrame({
        "event": event, "event_time": EVENT_TIMES,
        "did_pp": 100 * estimate, "se_pp": 100 * se,
        "ci_low_pp": 100 * (estimate - critical * se),
        "ci_high_pp": 100 * (estimate + critical * se),
        "p_value": pvalues, "matched_treated": n_total,
        "unique_controls": pair_table.control_hhid.nunique(),
        "household_clusters": g, "retained_cohorts": pair_table.cohort.nunique(),
        "is_reference": EVENT_TIMES == REFERENCE,
        "confidence_level": CONFIDENCE_LEVEL,
    })
    pre_idx = np.flatnonzero(EVENT_TIMES < REFERENCE)
    vpre = covariance[np.ix_(pre_idx, pre_idx)]
    rank = int(np.linalg.matrix_rank(vpre))
    # Do not report a seemingly valid joint test from a singular covariance.
    statistic, ppre = np.nan, np.nan
    if rank == len(pre_idx):
        bpre = estimate[pre_idx]
        statistic = float(bpre @ np.linalg.solve(vpre, bpre))
        ppre = float(stats.chi2.sf(statistic, rank))
    pretest = {"event": event, "wald_chi2": statistic, "df": rank,
               "pretrend_p_value": ppre, "full_rank": rank == len(pre_idx)}
    cov_rows = [{"event": event, "event_time_1": int(k), "event_time_2": int(l),
                 "covariance_pp_squared": 10000 * covariance[j, m]}
                for j, k in enumerate(EVENT_TIMES) for m, l in enumerate(EVENT_TIMES)]
    return rows, pretest, pd.DataFrame(cohort_rows), pd.DataFrame(cov_rows)


## 4. Plot six events with a common, data-driven Y axis
No smoothing or selective CI removal. Axis limits include every interval,
are symmetric around zero, and use about five major ticks. If an event has
no eligible matches, its panel says so. The orange line is at year 0;
-1 is the omitted reference, drawn as a hollow point at zero.


In [ ]:
def plot_event_study(results, output_dir):
    import tempfile
    # Avoid a slow/network-mounted default Matplotlib configuration directory.
    mpl_cache = Path(tempfile.gettempdir()) / "healthy_purchase_matplotlib"
    mpl_cache.mkdir(parents=True, exist_ok=True)
    os.environ["MPLCONFIGDIR"] = str(mpl_cache)
    global plt
    plt = load_module("matplotlib.pyplot")
    ticker = load_module("matplotlib.ticker")
    MaxNLocator, FuncFormatter = ticker.MaxNLocator, ticker.FuncFormatter
    plt.rcParams.update({"font.family": "serif", "font.size": 10,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "pdf.fonttype": 42, "ps.fonttype": 42})
    finite_ci = results[["ci_low_pp", "ci_high_pp"]].to_numpy().ravel() if len(results) else np.array([])
    finite_ci = finite_ci[np.isfinite(finite_ci)]
    bound = float(np.max(np.abs(finite_ci))) if finite_ci.size else 0.1
    bound = max(bound * 1.12, 0.01)
    locator = MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10], symmetric=True)
    ticks = locator.tick_values(-bound, bound)
    fig, axes = plt.subplots(2, 3, figsize=(12.6, 7.4), sharex=True, sharey=True)
    for ax, (event, label) in zip(axes.flat, EVENT_LABELS.items()):
        sub = results.loc[results.event.eq(event)].sort_values("event_time") if len(results) else pd.DataFrame()
        ax.axhline(0, color="#737373", linewidth=0.9, zorder=1)
        ax.axvline(0, color="#D58B34", linewidth=1.4, alpha=0.95, zorder=2)
        if len(sub):
            nonref = sub.loc[~sub.is_reference]
            ax.errorbar(nonref.event_time, nonref.did_pp,
                        yerr=np.vstack([nonref.did_pp - nonref.ci_low_pp,
                                        nonref.ci_high_pp - nonref.did_pp]),
                        fmt="o", color="#245A81", markersize=4.5,
                        elinewidth=1.15, capsize=3, zorder=4)
            ax.plot(sub.event_time, sub.did_pp, color="#245A81", linewidth=1.15, zorder=3)
            ax.scatter([REFERENCE], [0], facecolor="white", edgecolor="#245A81", s=29, zorder=5)
            n, c = int(sub.matched_treated.iloc[0]), int(sub.unique_controls.iloc[0])
            ax.text(0.03, 0.96, f"Treated: {n:,} | Controls: {c:,}",
                    transform=ax.transAxes, va="top", fontsize=8, color="#555555")
        else:
            ax.text(0.5, 0.5, "Insufficient eligible matches", ha="center", transform=ax.transAxes)
        ax.set_title(label, fontsize=11, pad=11)
        ax.set_xticks(EVENT_TIMES)
        ax.tick_params(axis="x", labelsize=14)
        ax.set_xlim(-PRE - 0.3, POST + 0.3)
        ax.set_yticks(ticks)
        ax.set_ylim(ticks[0], ticks[-1])
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.4f}".rstrip("0").rstrip(".") if abs(x) > 1e-10 else "0"))
        ax.grid(axis="y", color="#E6E6E6", linewidth=0.6)
        ax.set_axisbelow(True)
    fig.supylabel("Change in healthy food spending share (pp)", x=0.018, fontsize=11)
    fig.supxlabel("Years relative to event (reference: −1)", y=0.083, fontsize=16)
    fig.text(0.065, 0.023,
             "Matched DID; pointwise 90% household-clustered CIs conditional on matching. "
             "Orange line: event year.\n"
             "Equal weight per matched treated household; controls may be reused. "
             "Reference year −1 is normalized to zero.", fontsize=8, color="#555555")
    fig.subplots_adjust(left=0.085, right=0.985, top=0.94, bottom=0.18, wspace=0.12, hspace=0.32)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir / "matched_did_six_events.png", dpi=300, bbox_inches="tight")
    fig.savefig(output_dir / "matched_did_six_events.pdf", bbox_inches="tight")
    return fig


## 5. Run and save the V42 analysis

Outputs use a new V42 directory, preserving old DID results. Public outputs
contain only aggregate tables and figures; restricted annual observations and
matched household IDs stay outside GitHub. The pre-event outcome has changed,
so all six focal-event propensity models and matched samples are re-estimated.
Inspect audits, retention, balance, control reuse, and pretrend tests before
interpreting figures. This notebook is a specification update, not a claim
that the restricted NielsenIQ data were estimated in this environment.


In [ ]:
def run_analysis():
    table_dir, figure_dir = OUTPUT_DIR / "tables", OUTPUT_DIR / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    annual, audit = build_annual_panel(DATA_PATH, PRIVATE_DIR, table_dir)
    display(audit)
    summary = annual.groupby("panel_year").agg(
        households=("hhid", "nunique"), trips=("n_trips", "sum"),
        valid_outcome_households=("Y", "count"),
        mean_annual_share=("Y", "mean"), median_annual_share=("Y", "median"),
    ).reset_index()
    summary.to_csv(table_dir / "annual_sample_summary.csv", index=False)
    display(summary)
    ywide = annual.pivot(index="hhid", columns="panel_year", values="Y").reindex(
        columns=range(YEAR_START, YEAR_END + 1))
    results, diagnostics, balances, pretests, cohort_results, covariances = [], [], [], [], [], []
    statuses = []
    matched_pairs = {}  # Frozen once; reused for observed and model-implied rates.
    for event, label in EVENT_LABELS.items():
        print(f"Matching: {label} ...", flush=True)
        pairs, diag, balance = match_event(annual, event, ywide)
        matched_pairs[event] = pairs.copy()
        diagnostics.append(diag)
        if not balance.empty:
            balances.append(balance)
        if pairs.empty:
            statuses.append({"event": event, "status": "no_retained_cohort", "matched_treated": 0})
            print("  No cohort met the matching thresholds.")
            continue
        if SAVE_PRIVATE_MATCHES:
            match_dir = PRIVATE_DIR / "matched_household_ids"
            match_dir.mkdir(parents=True, exist_ok=True)
            pairs.to_parquet(match_dir / f"{event}.parquet", index=False)
        estimates, pretest, cohort_table, covariance = estimate_event(pairs, ywide, event)
        results.append(estimates)
        pretests.append(pretest)
        cohort_results.append(cohort_table)
        covariances.append(covariance)
        statuses.append({"event": event, "status": "estimated", "matched_treated": len(pairs),
                         "unique_controls": pairs.control_hhid.nunique(),
                         "max_control_reuse": int(pairs.control_hhid.value_counts().max()),
                         "control_effective_sample_size": len(pairs)**2 / float((pairs.control_hhid.value_counts()**2).sum())})
        print(f"  Retained {len(pairs):,} treated households; {pairs.control_hhid.nunique():,} unique controls.")
    result_table = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=["event"])
    balance_table = pd.concat(balances, ignore_index=True) if balances else pd.DataFrame(
        columns=["event", "cohort", "feature", "smd_before", "smd_after", "absolute_smd_after", "balance_flag"])
    diagnostic_table = pd.concat(diagnostics, ignore_index=True)
    files = {
        "event_study_coefficients.csv": result_table,
        "matching_sample_flow.csv": diagnostic_table,
        "matching_balance.csv": balance_table,
        "event_status.csv": pd.DataFrame(statuses),
        "pretrend_joint_tests.csv": pd.DataFrame(pretests, columns=["event", "wald_chi2", "df", "pretrend_p_value", "full_rank"]),
        "cohort_event_study.csv": pd.concat(cohort_results, ignore_index=True) if cohort_results else pd.DataFrame(columns=["event"]),
        "event_study_covariance.csv": pd.concat(covariances, ignore_index=True) if covariances else pd.DataFrame(columns=["event"]),
    }
    for filename, frame in files.items():
        # Guard against accidental household-ID exports to the public folder.
        if any("hhid" in str(c).lower() for c in frame.columns):
            raise RuntimeError(f"Household identifier in aggregate export: {filename}")
        frame.to_csv(table_dir / filename, index=False)
    if not balance_table.empty:
        balance_summary = balance_table.groupby("event").agg(
            max_absolute_smd_after=("absolute_smd_after", "max"),
            flagged_feature_cohort_cells=("balance_flag", "sum"),
        ).reset_index()
        balance_summary.to_csv(table_dir / "matching_balance_summary.csv", index=False)
        display(balance_summary)
        if balance_table.balance_flag.any():
            warnings.warn("Some matched covariates have |SMD| > 0.10 or undefined balance; inspect matching_balance.csv.")
    config = {
        "pipeline_version": PIPELINE_VERSION, "input_filename": DATA_PATH.name,
        "confidence_level": CONFIDENCE_LEVEL,
        "milk_is_separate_category": False,
        "five_categories": RESEARCH_CATEGORIES,
        "source_milk_y_rule": "1 or 2 implies ordinary milk is in the healthy beverage category",
        "healthy_spending_columns": HEALTHY_SPENDING_COLUMNS,
        "healthy_columns_by_outcome": HEALTHY_COLUMNS,
        "outcome_allowed_codes": OUTCOME_ALLOWED_CODES,
        "healthy_yogurt_definition": "regular sugar; high sugar excluded",
        "healthy_cereal_definition": "low plus medium sugar; high sugar excluded",
        "combined_beverage_healthy_spending": ["healthy_beverage_spending", *MILK_SPENDING_COLUMNS],
        "denominator": "sum of trip food_spend within household-year",
        "year_start": YEAR_START, "year_end": YEAR_END,
        "event_window": [-PRE, POST], "reference": REFERENCE,
        "matching": "1:1 nearest logit propensity score, with replacement, within calendar cohort",
        "propensity_C": 1000, "baseline_features": [f"Y_pre{k}" for k in range(PRE, 0, -1)] + COVARIATES,
        "exact_region_at_minus_one": EXACT_REGION, "caliper_pooled_logit_sd": CALIPER_SD,
        "controls": "no focal event observed anywhere in input panel",
        "treated": "first observed focal event; no repeat within complete event window",
        "minimum_eligible_treated": MIN_TREATED, "minimum_eligible_controls": MIN_CONTROLS,
        "minimum_retained_pairs_per_cohort": MIN_MATCHED,
        "duplicate_trip_check": CHECK_DUPLICATE_TRIPS,
        "aggregation_mode": AGGREGATION_MODE, "duckdb_threads": DUCKDB_THREADS,
        "inference": "fixed-match cohort-mean sandwich clustered by original household; G/(G-1) correction; pointwise t CIs",
        "first_stage_uncertainty_included": False,
        "versions": {p: importlib.metadata.version(p) for p in
                     ["duckdb", "pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "pyarrow"]},
    }
    (OUTPUT_DIR / "analysis_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "README.md").write_text(
        "# Matched DID event study\n\n"
        "Outcome: annual healthy-subcategory dollars / annual sum of trip food_spend. "
        "See analysis_config.json for category inclusion and matching settings.\n\n"
        "V42 uses regular-sugar yogurt spending and low- plus medium-sugar cereal spending. "
        "High-sugar yogurt and cereal spending are excluded from the healthy numerator. "
        "All existing ordinary milk (skimmed, low-fat, other) enters the healthy "
        "beverage spending numerator, counting each source column exactly once. "
        "Milk is not a separate research category. Source outcome labels are not "
        "used as spending weights. The six life-event analyses use first observed events, "
        "no focal-event recurrence in the window, "
        "never-observed-event controls, complete outcome windows, and pre-event matching only. "
        "The analysis estimates dynamic matched changes relative to year -1, in percentage points. "
        "The orange vertical line marks year 0. These are adjusted associations; "
        "causal interpretation requires additional assumptions.\n\n"
        "Pointwise 90% CIs use household-clustered fixed-match sandwich SEs. "
        "Control reuse across cohorts is included; propensity estimation and matching uncertainty are not. "
        "Pretrend tests are partly constrained by matching on pre-event outcomes. "
        "Full-window restrictions select on observation/attrition.\n\n"
        "Inspect tables/matching_sample_flow.csv and tables/matching_balance.csv before interpreting figures. "
        "Counts are unweighted household counts; no Nielsen projection weights are used. "
        "Annual mean shares are unweighted household means, not aggregate dollar market shares.\n\n"
        "This folder contains aggregate outputs only. Annual microdata and matched household IDs "
        "are saved separately under the private data directory.\n",
        encoding="utf-8")
    fig = plot_event_study(result_table, figure_dir)
    display(pd.DataFrame(statuses))
    display(pd.DataFrame(pretests))
    plt.show()
    print(f"Finished. Aggregate outputs: {OUTPUT_DIR}")
    return {"annual_panel": annual, "coefficients": result_table,
            "sample_flow": diagnostic_table, "balance": balance_table, "figure": fig,
            "matched_pairs": matched_pairs}


print("[START] Full analysis: audit -> annual aggregation -> matching -> figures", flush=True)
analysis = run_analysis()


## 6. Observed versus model-implied healthy-spending DID

**Same outcome, same matches:** the observed curve must reproduce the original
spending DID exactly. The comparison now predicts healthy dollars, replacing the
former any-healthy-trip purchase-rate comparison.

For trip $r$ in household $h$, year $t$ and category $c$, let $B_{htrc}$ be
actual category spending, $A_{htrc}$ actual healthy spending, and
$R_{htrc}=A_{htrc}/B_{htrc}$ for $B>0$. Source dollars are:

| Category | Actual category budget $B$ | Healthy dollars $A$ |
| --- | --- | --- |
| FV | fresh + non-fresh | fresh |
| Yogurt | regular sugar + high sugar | regular sugar |
| Cereal | low + medium + high sugar | low + medium sugar |
| Snacks | healthy + unhealthy | healthy |
| Combined beverages | healthy + unhealthy drinks + all three milk columns | healthy drinks + all three milk columns |

No source column is counted twice. All ordinary milk remains in healthy
beverages. Each trip-category's actual budget is fixed, including zero budgets.

### Auxiliary amount allocation

For each original purchased-category outcome $k$, estimate a **state-independent**
fractional-logit conditional mean of $R$ using positive-budget trips in the HMM
estimation sample. Predictors are children (`K_1`), income, household size, age,
and fixed category-budget bands: (0,2], (2,5], (5,10], (10,20], and >20 dollars.
No life-event or calendar-year coefficients are added to this auxiliary model.
There is one fit per category/outcome/fold: 2 FV, 2 yogurt, 3 cereal, 2 snack,
and 2 beverage outcomes. Separate low- and medium-cereal amount fractions are
combined with the corresponding sequential-gate probabilities.

Five deterministic **household folds** keep all years of a household together.
Each matched household's auxiliary amount fraction is predicted using a fit
that excluded that household entirely. This cross-fitting applies to the amount
equations; the HMM coefficients and smoothed posteriors retain their existing
within-sample interpretation. A small declared ridge penalty stabilizes the
auxiliary prediction; the intercept is unpenalized. All-zero/all-one training
fractions use exact boundary constants. Missing category/outcome training support
stops the comparison rather than assigning an arbitrary amount fraction.

The loss uses **one weight per trip**, not per dollar or matched appearance.
Household-year/outcome/budget-band counts and sums of $R$ are exact sufficient
statistics for the specified grouped predictors. Private caches avoid repeating
Parquet scans on subsequent runs. The auxiliary tables report prediction
coefficients and numerical diagnostics, not coefficient significance.

Fractional response mean models retain outcomes at 0 and 1; see
[Papke and Wooldridge (1996)](https://www.nber.org/papers/t0147).
The ridge penalty and household cross-fitting are explicit prediction choices
in this implementation, not claims about that paper's original estimator.

### Expected healthy dollars

Let $\gamma_{ht,s}$ be the original smoothed HMM posterior, and let
$P_{c,k,s}(x_{ht})$ be its conditional purchased-category choice probabilities.
The auxiliary mean $\hat m^{(-fold(h))}_{c,k}(x,b)$ conditions on the hypothetical
model outcome, household controls, and the actual budget band:

$$\widehat A_{htrc}=B_{htrc}\sum_{s=1}^{2}\gamma_{ht,s}
\sum_{k=1}^{K_c}P_{c,k,s}(x_{ht})
\hat m^{(-fold(h))}_{c,k}(x_{ht},band(B_{htrc})).$$

Predictions sum over **all possible model choices**; they do not condition on the
matched trip's observed choice code when allocating its budget. For cereal:
$P_1=p_{low}$, $P_2=(1-p_{low})p_{medium\mid not\ low}$, and
$P_3=(1-p_{low})(1-p_{medium\mid not\ low})$. Source HMM state coding is 1 and 2.

**Do not multiply by category incidence again:** the actual category budget
already determines whether that category is purchased. A zero budget contributes
zero dollars. A mixed basket can receive a fraction strictly between zero and
one, even when its choice code is the healthy alternative.

$$Y^{obs}_{ht}=\frac{\sum_{r,c}A_{htrc}}{\sum_r food\_spend_{htr}},\qquad
Y^{model}_{ht}=\frac{\sum_{r,c}\widehat A_{htrc}}{\sum_r food\_spend_{htr}}.$$

Both curves use the identical observed annual food-spending denominator, frozen
matched pairs, cohort weights, control reuse and year -1 reference. Audits require
the observed numerator and DID estimates to reproduce the main spending analysis.
There is no HMM re-estimation and no change to its likelihood, AIC or BIC. Auxiliary
parameters are additional post-estimation predictions and are not counted as HMM
parameters. Missing matched model coverage or invalid budgets stop the comparison.

### Interpretation and outputs

This is a **conditional spending-allocation fit**, given actual category budgets,
household covariates and smoothed purchase histories. Total category spending and
event/no-event budget changes are not predicted. Its fit cannot by itself validate
a fully structural event effect; neither curve's direction is constrained.

The observed curve has **90% fixed-match household-clustered intervals**. The model
curve shows point predictions. CSV conditional sandwich calculations hold HMM,
auxiliary fits and posteriors fixed; they do not include their estimation
uncertainty or matching uncertainty. No HMM/auxiliary confidence band is drawn.

Run the supplied V42 HMM once and set `MODEL_RESULTS_DIR` to the folder containing
`run_configuration.json` and `02_proposed_hmm`. After running the main spending
analysis, run the two cells below. The auxiliary calibration runs here.

The combined six-event figure is
`figures/matched_did_healthy_spending_observed_vs_hmm.png` (and PDF).
Aggregate CSVs use `healthy_spending_*.csv`, including auxiliary parameters,
diagnostics, exact-match audits and DID coefficients. Household/year and category
predictions plus calibration caches are kept only in the private data directory.


In [ ]:
def spending_model_inputs(results_dir, data_path):
    """Read point estimates/posteriors from the supplied V42 HMM's CSV exports."""
    results_dir = Path(results_dir)
    folder = results_dir / "02_proposed_hmm"
    paths = {
        "configuration": results_dir / "run_configuration.json",
        "coefficients": folder / "01_full_coefficient_table.csv",
        "posterior": folder / "06_posterior_household_year.csv",
        "fit": folder / "05_model_fit.csv",
    }
    missing = [str(p) for p in paths.values() if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            "Run the V42 HMM first, then set MODEL_RESULTS_DIR to its sample folder "
            "(the folder containing run_configuration.json and 02_proposed_hmm). "
            "Missing: " + "; ".join(missing))
    config = json.loads(paths["configuration"].read_text(encoding="utf-8"))
    categories = ["fv", "yogurt", "cereal", "snack", "beverage"]
    covariates = ["K_1", "income10k_imputed", "family_size_numeric", "age"]
    if config.get("categories") != categories or config.get("choice_covariates") != covariates:
        raise ValueError("These prediction equations require the supplied five-category V42 choice HMM.")
    if config.get("version") != "V42" or config.get("run_mode") != "full_estimation":
        raise ValueError("Use full-estimation V42 HMM outputs, not synthetic/validation outputs.")
    if Path(config["input_path"]).resolve() != Path(data_path).resolve():
        raise ValueError("The HMM and DID input paths differ; use estimates from the same sugar panel.")
    if "regular sugar" not in config.get("yogurt_definition", "").lower():
        raise ValueError("The model must use regular-sugar versus high-sugar yogurt.")
    coefficients = pd.read_csv(paths["coefficients"])
    if coefficients.parameter.duplicated().any():
        raise ValueError("Duplicate model parameter names.")
    beta = coefficients.set_index("parameter")["estimate"]
    if not np.isfinite(beta.to_numpy(dtype=float)).all():
        raise ValueError("Nonfinite model coefficients.")
    gamma = pd.read_csv(paths["posterior"], dtype={"hhid": str})
    required = ["hhid", "panel_year", "gamma_state_1", "gamma_state_2"]
    if set(required) - set(gamma):
        raise ValueError(f"Missing posterior columns: {set(required) - set(gamma)}")
    gamma = gamma[required].copy()
    gamma["panel_year"] = pd.to_numeric(gamma.panel_year, errors="raise")
    values = gamma[["gamma_state_1", "gamma_state_2"]].to_numpy(dtype=float)
    if (gamma[required].isna().any().any() or gamma.duplicated(["hhid", "panel_year"]).any()
            or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any()
            or not np.allclose(values.sum(axis=1), 1, atol=1e-8, rtol=0)):
        raise ValueError("Invalid, duplicate, or unnormalized household-year posteriors.")
    fit = pd.read_csv(paths["fit"])
    if len(fit) != 1 or fit.model.iloc[0] != "proposed_hmm":
        raise ValueError("Expected one proposed_hmm fit record.")
    if str(fit.converged.iloc[0]).lower() != "true":
        warnings.warn("The supplied HMM fit reports nonconvergence; interpret its fitted curve accordingly.")
    return beta, gamma, config, paths


def frozen_match_window_keys(matched_pairs):
    """The union of exactly the household-years used by the original matches."""
    parts = []
    for pairs in matched_pairs.values():
        if pairs.empty:
            continue
        for cohort, group in pairs.groupby("cohort", sort=True):
            ids = pd.unique(pd.concat([group.treated_hhid, group.control_hhid], ignore_index=True))
            parts.append(pd.MultiIndex.from_product(
                [ids, int(cohort) + EVENT_TIMES], names=["hhid", "panel_year"]).to_frame(index=False))
    if not parts:
        return pd.DataFrame(columns=["hhid", "panel_year"])
    keys = pd.concat(parts, ignore_index=True).drop_duplicates()
    keys["hhid"] = keys.hhid.astype(str)
    if keys.duplicated(["hhid", "panel_year"]).any():
        raise ValueError("Household IDs collided after string conversion.")
    return keys.sort_values(["hhid", "panel_year"]).reset_index(drop=True)


def conditional_choice_weights(annual, beta):
    """Posterior-weighted probabilities of each purchased-category outcome."""
    expit = load_module("scipy.special").expit
    covariates = ["K_1", "income10k_imputed", "family_size_numeric", "age"]
    x = annual[covariates].to_numpy(dtype=float)
    if not np.isfinite(x).all():
        raise ValueError("Nonfinite choice covariates on matched household-years.")
    x = np.column_stack([np.ones(len(x)), x])
    gates = dict(fv="fv_fresh", yogurt="yogurt_regular_sugar", cereal="cereal_low_sugar",
                 snack="snack_healthy", beverage="beverage_healthier")
    result = {}
    for category, gate in gates.items():
        for state in (1, 2):
            def probability(name):
                names = [f"{name}_gate_{v}" for v in ["intercept", "state_num", *covariates]]
                if set(names) - set(beta.index):
                    raise ValueError(f"Missing choice parameters: {set(names) - set(beta.index)}")
                b = beta.loc[names].to_numpy(dtype=float)
                # Source V42 uses state_num=1 and state_num=2.
                return expit(x @ np.r_[b[0], b[2:]] + state*b[1])
            first = probability(gate)
            outcomes = {1: first, 2: 1-first}
            if category == "cereal":
                medium = probability("cereal_medium_given_notlow")
                outcomes = {1: first, 2: (1-first)*medium, 3: (1-first)*(1-medium)}
            gamma = annual[f"gamma_state_{state}"].to_numpy(dtype=float)
            for code, values in outcomes.items():
                key = f"weight_{category}_{code}"
                result[key] = result.get(key, 0) + gamma*values
    return pd.DataFrame(result, index=annual.index)


def amount_category_specs():
    """Disjoint source dollars, and the original choice codes used by V42."""
    return {
        "fv": dict(code="t.fv_y", codes=[1, 2], healthy=["fresh_spending"],
                   components={"fresh_spending": "fv_y", "non_fresh_spending": "fv_y"}),
        "yogurt": dict(code="t.yogurt_y", codes=[1, 2], healthy=["regular_sugar_spending"],
                       components={"regular_sugar_spending": "yogurt_y", "high_sugar_spending": "yogurt_y"}),
        "cereal": dict(code="t.cereal_y", codes=[1, 2, 3],
                       healthy=["low_sugar_cereal_spending", "medium_sugar_cereal_spending"],
                       components={c: "cereal_y" for c in ["low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending"]}),
        "snack": dict(code="t.snack_y", codes=[1, 2], healthy=["healthy_snack_spending"],
                      components={"healthy_snack_spending": "snack_y", "unhealthy_snack_spending": "snack_y"}),
        "beverage": dict(code="CASE WHEN t.milk_y IN (1,2) OR t.beverage_y=1 THEN 1 WHEN t.beverage_y=2 THEN 2 ELSE 0 END",
                         codes=[1, 2], healthy=["healthy_beverage_spending", *MILK_SPENDING_COLUMNS],
                         components={"healthy_beverage_spending": "beverage_y", "unhealthy_beverage_spending": "beverage_y",
                                     **{c: "milk_y" for c in MILK_SPENDING_COLUMNS}}),
    }


def auxiliary_household_folds(ids):
    """Stable household-level folds: all years of one household stay together."""
    import hashlib
    if type(AMOUNT_AUX_FOLDS) is not int or AMOUNT_AUX_FOLDS < 2:
        raise ValueError("AMOUNT_AUX_FOLDS must be an integer >= 2.")
    unique = pd.Index(pd.unique(ids.astype(str)))
    mapping = {hh: int.from_bytes(hashlib.blake2b(
        f"{AMOUNT_AUX_SEED}:{hh}".encode(), digest_size=8).digest(), "little") % AMOUNT_AUX_FOLDS
        for hh in unique}
    return ids.astype(str).map(mapping).astype(int)


def amount_aux_design(frame):
    """Annual controls plus fixed budget-band indicators; no events/year effects."""
    controls = ["K_1", "income10k_imputed", "family_size_numeric", "age"]
    columns = [frame[c].to_numpy(dtype=float) for c in controls]
    columns += [frame.budget_band.eq(b).to_numpy(dtype=float) for b in range(1, len(AMOUNT_BUDGET_CUTS)+1)]
    names = controls + [f"budget_band_{b}" for b in range(1, len(AMOUNT_BUDGET_CUTS)+1)]
    x = np.column_stack(columns)
    if not np.isfinite(x).all():
        raise ValueError("Nonfinite auxiliary amount predictors.")
    return x, names


def fit_amount_auxiliary(train, category, code, heldout_fold):
    """Grouped fractional logit; exact compression for the specified predictors.

    Each original positive-budget trip has weight one. Grouped counts/sums
    exactly reproduce its fractional Bernoulli quasi-loss. A small, declared
    ridge penalty stabilizes prediction; no coefficient significance is claimed.
    """
    expit = load_module("scipy.special").expit
    minimize = load_module("scipy.optimize").minimize
    if train.empty:
        raise ValueError(f"No training trips for {category}, code {code}, heldout fold {heldout_fold}. "
                         "The conditional amount mean is unidentified; no automatic imputation is used.")
    n = train.n_positive_budget_trips.to_numpy(dtype=float)
    success = train.sum_healthy_fraction.to_numpy(dtype=float)
    total = float(n.sum())
    mean = float(success.sum()/total)
    diagnostic = dict(category=category, choice_code=code, heldout_fold=heldout_fold,
                      training_trips=int(total), training_households=train.hhid.nunique(),
                      training_group_rows=len(train), training_mean_fraction=mean,
                      ridge_penalty=AMOUNT_AUX_RIDGE, training_holdout_household_overlap=0)
    if mean <= 1e-14 or mean >= 1-1e-14:
        constant = 0.0 if mean <= 1e-14 else 1.0
        diagnostic.update(method="boundary_constant", converged=True, iterations=0, gradient_max_abs=0.0)
        params = [dict(category=category, choice_code=code, heldout_fold=heldout_fold,
                       parameter="constant_share", estimate=constant)]
        return dict(constant=constant), diagnostic, params
    if not 0 <= AMOUNT_AUX_RIDGE < 1:
        raise ValueError("AMOUNT_AUX_RIDGE must lie in [0,1).")
    raw, names = amount_aux_design(train)
    center = (n @ raw)/total
    scale = np.sqrt((n @ ((raw-center)**2))/total)
    active = scale > 1e-10
    design = np.column_stack([np.ones(len(train)), (raw[:, active]-center[active])/scale[active]])
    y = success/n
    weight = n/total
    def objective(b):
        eta = design @ b
        loss = np.sum(weight*(np.logaddexp(0, eta)-y*eta)) + 0.5*AMOUNT_AUX_RIDGE*(b[1:] @ b[1:])
        gradient = design.T @ (weight*(expit(eta)-y))
        gradient[1:] += AMOUNT_AUX_RIDGE*b[1:]
        return float(loss), gradient
    start = np.zeros(design.shape[1])
    start[0] = np.log(mean)-np.log1p(-mean)
    fitted = minimize(objective, start, jac=True, method="L-BFGS-B",
                      options=dict(maxiter=300, ftol=1e-12, gtol=1e-7))
    _, gradient = objective(fitted.x)
    converged = bool(fitted.success or np.max(np.abs(gradient)) < 1e-6)
    diagnostic.update(method="ridge_fractional_logit", converged=converged,
                      iterations=int(fitted.nit), gradient_max_abs=float(np.max(np.abs(gradient))))
    if not converged or not np.isfinite(fitted.x).all():
        raise RuntimeError(f"Auxiliary fit failed for {category}/{code}/fold {heldout_fold}: {fitted.message}")
    slopes = np.zeros(raw.shape[1])
    slopes[active] = fitted.x[1:]/scale[active]
    intercept = float(fitted.x[0]-slopes@center)
    params = [dict(category=category, choice_code=code, heldout_fold=heldout_fold,
                   parameter=name, estimate=float(value))
              for name, value in zip(["intercept", *names], [intercept, *slopes])]
    return dict(intercept=intercept, slopes=slopes), diagnostic, params


def predict_amount_auxiliary(fitted, frame):
    if "constant" in fitted:
        return np.full(len(frame), fitted["constant"])
    expit = load_module("scipy.special").expit
    x, _ = amount_aux_design(frame)
    return expit(fitted["intercept"] + x @ fitted["slopes"])


def amount_category_statistics(con, category, spec, all_hmm_years, private_dir):
    """Audit/aggregate one category; optional cache contains restricted statistics."""
    import hashlib
    cuts = np.asarray(AMOUNT_BUDGET_CUTS, dtype=float)
    if (not np.isfinite(cuts).all() or (cuts <= 0).any() or (np.diff(cuts) <= 0).any()):
        raise ValueError("AMOUNT_BUDGET_CUTS must be finite, positive, and increasing.")
    identity = pd.util.hash_pandas_object(all_hmm_years[["hhid", "panel_year"]], index=False).to_numpy().tobytes()
    stamp = dict(version="V42-fixed-budget-amount-v1", source=str(DATA_PATH.resolve()),
                 size=DATA_PATH.stat().st_size, mtime_ns=DATA_PATH.stat().st_mtime_ns,
                 model_keys_sha256=hashlib.sha256(identity).hexdigest(),
                 category=category, spec=spec, budget_cuts=list(AMOUNT_BUDGET_CUTS))
    cache = Path(private_dir) / f"amount_aux_statistics_{category}.parquet"
    metadata = cache.with_suffix(".json")
    if USE_AMOUNT_CALIBRATION_CACHE and cache.is_file() and metadata.is_file():
        saved = json.loads(metadata.read_text(encoding="utf-8"))
        if saved.get("stamp") == stamp:
            return pd.read_parquet(cache), saved["audit"]
    required = set(spec["components"])
    actual = {r[0] for r in con.execute("DESCRIBE amount_source").fetchall()}
    if required-actual:
        raise ValueError(f"Missing {category} amount columns: {sorted(required-actual)}")
    budget = " + ".join(f"COALESCE(CAST(t.{qi(c)} AS DOUBLE),0.0)" for c in spec["components"])
    healthy = " + ".join(f"COALESCE(CAST(t.{qi(c)} AS DOUBLE),0.0)" for c in spec["healthy"])
    invalid = " OR ".join(
        f"(t.{qi(c)} IS NULL AND t.{qi(owner)}>0) OR "
        f"(t.{qi(c)} IS NOT NULL AND (NOT isfinite(CAST(t.{qi(c)} AS DOUBLE)) OR t.{qi(c)}<0))"
        for c, owner in spec["components"].items())
    con.execute("CREATE OR REPLACE TEMP VIEW amount_category_trips AS SELECT "
                f"CAST(t.hhid AS VARCHAR) AS hhid, t.panel_year, {spec['code']} AS choice_code, "
                f"({budget}) AS budget, ({healthy}) AS healthy, ({invalid}) AS invalid_amount "
                "FROM amount_source t JOIN amount_hmm_keys k "
                "ON CAST(t.hhid AS VARCHAR)=k.hhid AND t.panel_year=k.panel_year")
    allowed = ",".join(map(str, [0, *spec["codes"]]))
    query = f"""SELECT COUNT(*) AS trip_rows,
        COUNT(*) FILTER (WHERE invalid_amount) AS invalid_amount_rows,
        COUNT(*) FILTER (WHERE choice_code IS NULL OR choice_code NOT IN ({allowed})) AS invalid_code_rows,
        COUNT(*) FILTER (WHERE budget>0 AND choice_code=0) AS positive_budget_nonpurchase_rows,
        COUNT(*) FILTER (WHERE healthy>budget+1e-8*greatest(1.0,budget)) AS excess_healthy_rows,
        COUNT(*) FILTER (WHERE budget=0 AND choice_code>0) AS zero_dollar_purchase_rows
        FROM amount_category_trips"""
    with progress(f"Audit {category} category budgets"):
        raw = con.execute(query).fetchdf().iloc[0]
    audit = {k: int(v) for k, v in raw.items()}
    if any(audit[k] for k in ["invalid_amount_rows", "invalid_code_rows", "positive_budget_nonpurchase_rows", "excess_healthy_rows"]):
        pd.DataFrame([dict(category=category, **audit)]).to_csv(
            Path(private_dir)/f"amount_failed_audit_{category}.csv", index=False)
        raise ValueError(f"Invalid {category} budgets/codes; see the private amount_failed_audit file.")
    band = "CASE " + " ".join(f"WHEN budget<={float(edge):.17g} THEN {i}" for i, edge in enumerate(cuts)) + f" ELSE {len(cuts)} END"
    with progress(f"Compress {category} amounts by household-year, choice and budget band"):
        grouped = con.execute(
            f"SELECT hhid, panel_year, choice_code, {band} AS budget_band, "
            "COUNT(*) AS n_positive_budget_trips, SUM(budget) AS category_budget, "
            "SUM(healthy) AS observed_healthy_spend, "
            "SUM(LEAST(1.0,GREATEST(0.0,healthy/budget))) AS sum_healthy_fraction "
            "FROM amount_category_trips WHERE budget>0 GROUP BY hhid,panel_year,choice_code,budget_band").fetchdf()
    grouped.to_parquet(cache, index=False)
    metadata.write_text(json.dumps(dict(stamp=stamp, audit=audit), indent=2), encoding="utf-8")
    return grouped, audit


def build_matched_healthy_spending(analysis, model_results_dir, private_dir, table_dir):
    """Fixed observed category budgets times cross-fitted expected healthy fractions."""
    private_dir, table_dir = Path(private_dir), Path(table_dir)
    private_dir.mkdir(parents=True, exist_ok=True); table_dir.mkdir(parents=True, exist_ok=True)
    beta, gamma, config, paths = spending_model_inputs(model_results_dir, DATA_PATH)
    annual = analysis["annual_panel"].copy()
    annual["hhid"] = annual.hhid.astype(str)
    if annual.duplicated(["hhid", "panel_year"]).any():
        raise ValueError("Duplicate annual keys after household-ID string conversion.")
    needed = frozen_match_window_keys(analysis["matched_pairs"])
    matched = needed.merge(annual, on=["hhid", "panel_year"], how="left", validate="one_to_one", indicator=True)
    if not matched._merge.eq("both").all():
        raise ValueError("A frozen matched window is missing from the original annual panel.")
    matched = matched.drop(columns="_merge").merge(gamma, on=["hhid", "panel_year"],
                                                how="left", validate="one_to_one", indicator=True)
    missing = matched.loc[matched._merge.ne("both"), ["hhid", "panel_year"]]
    if len(missing):
        missing.to_csv(private_dir/"missing_hmm_matched_household_years.csv", index=False)
        raise ValueError(f"{len(missing)} matched household-years lack HMM posteriors; no pairs were dropped.")
    matched = matched.drop(columns="_merge")
    matched = pd.concat([matched, conditional_choice_weights(matched, beta)], axis=1)
    matched["household_fold"] = auxiliary_household_folds(matched.hhid)
    all_hmm = annual.merge(gamma[["hhid", "panel_year"]], on=["hhid", "panel_year"], how="inner", validate="one_to_one")
    all_hmm["household_fold"] = auxiliary_household_folds(all_hmm.hhid)
    if all_hmm.household_fold.nunique() < 2:
        raise ValueError("At least two nonempty household folds are required for auxiliary cross-fitting.")
    controls = ["K_1", "income10k_imputed", "family_size_numeric", "age"]
    if not np.isfinite(all_hmm[controls].to_numpy(dtype=float)).all():
        raise ValueError("Nonfinite controls in the auxiliary training sample.")
    all_hmm = all_hmm.sort_values(["hhid", "panel_year"]).reset_index(drop=True)
    specs = amount_category_specs()
    dollars = matched[["hhid", "panel_year", "n_trips", "food_spend", "healthy_spend", "Y"]].copy()
    category_outputs, diagnostics, parameters, amount_audits = [], [], [], []
    source_sql = DATA_PATH.as_posix().replace("'", "''")
    con = duckdb.connect()
    try:
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        con.execute(f"SET threads={int(DUCKDB_THREADS)}")
        temp_sql = (private_dir/"amount_comparison_duckdb_temp").as_posix().replace("'", "''")
        con.execute(f"SET temp_directory='{temp_sql}'")
        con.execute(f"CREATE TEMP VIEW amount_source AS SELECT * FROM read_parquet('{source_sql}')")
        con.register("amount_hmm_keys", all_hmm[["hhid", "panel_year"]])
        for category, spec in specs.items():
            grouped, audit = amount_category_statistics(con, category, spec, all_hmm, private_dir)
            if audit["trip_rows"] != int(all_hmm.n_trips.sum()):
                raise ValueError("Source and HMM-sample trip counts disagree.")
            amount_audits.append(dict(category=category, **audit))
            train = grouped.merge(all_hmm[["hhid", "panel_year", "household_fold", *controls]],
                                  on=["hhid", "panel_year"], how="left", validate="many_to_one")
            if train.household_fold.isna().any():
                raise ValueError("An auxiliary training group has no annual covariates/fold.")
            # Drop the observed outcome before prediction: budgets are held fixed,
            # but the HMM reassigns all possible purchased-category choices.
            budget = grouped.groupby(["hhid", "panel_year", "budget_band"], as_index=False).agg(
                category_budget=("category_budget", "sum"), observed_healthy_spend=("observed_healthy_spend", "sum"))
            weight_cols = [f"weight_{category}_{code}" for code in spec["codes"]]
            predict = budget.merge(matched[["hhid", "panel_year", "household_fold", *controls, *weight_cols]],
                                   on=["hhid", "panel_year"], how="inner", validate="many_to_one")
            if predict.empty:
                output = matched[["hhid", "panel_year"]].copy()
                output["category_budget"] = output["observed_healthy_spend"] = output["model_healthy_spend"] = 0.0
            else:
                predict["model_fraction"] = 0.0
                for fold in sorted(predict.household_fold.unique()):
                    test_mask = predict.household_fold.eq(fold)
                    test = predict.loc[test_mask]
                    training = train.loc[train.household_fold.ne(fold)]
                    if set(test.hhid) & set(training.hhid):
                        raise RuntimeError("Household leakage across auxiliary folds.")
                    for code in spec["codes"]:
                        with progress(f"Fit {category} amount fraction, code {code}, held-out fold {fold}"):
                            fitted, diagnostic, params = fit_amount_auxiliary(
                                training.loc[training.choice_code.eq(code)], category, code, int(fold))
                        fraction = predict_amount_auxiliary(fitted, test)
                        predict.loc[test_mask, "model_fraction"] += test[f"weight_{category}_{code}"].to_numpy()*fraction
                        diagnostics.append(diagnostic); parameters.extend(params)
                predict["model_healthy_spend"] = predict.category_budget*predict.model_fraction
                output = predict.groupby(["hhid", "panel_year"], as_index=False)[
                    ["category_budget", "observed_healthy_spend", "model_healthy_spend"]].sum()
                output = matched[["hhid", "panel_year"]].merge(output, on=["hhid", "panel_year"], how="left", validate="one_to_one")
                output[["category_budget", "observed_healthy_spend", "model_healthy_spend"]] = output[
                    ["category_budget", "observed_healthy_spend", "model_healthy_spend"]].fillna(0.0)
            output["category"] = category
            category_outputs.append(output)
            del grouped, train, budget, predict
    finally:
        con.close()
    category_table = pd.concat(category_outputs, ignore_index=True)
    totals = category_table.groupby(["hhid", "panel_year"], as_index=False).agg(
        category_budget=("category_budget", "sum"), observed_healthy_spend=("observed_healthy_spend", "sum"),
        model_healthy_spend=("model_healthy_spend", "sum"))
    dollars = dollars.merge(totals, on=["hhid", "panel_year"], how="left", validate="one_to_one")
    if (dollars.food_spend <= 0).any() or not np.isfinite(dollars.food_spend.to_numpy()).all():
        raise ValueError("Invalid annual food-spending denominator on the frozen matched sample.")
    if not np.allclose(dollars.observed_healthy_spend, dollars.healthy_spend, rtol=1e-8, atol=1e-7):
        raise ValueError("Observed healthy dollars do not reproduce the main spending DID numerator.")
    if (dollars.category_budget > dollars.food_spend + 1e-8*np.maximum(1, dollars.food_spend)).any():
        raise ValueError("The five category budgets exceed annual food_spend; check source category overlap.")
    dollars["observed_share"] = dollars.observed_healthy_spend/dollars.food_spend
    dollars["model_implied_share"] = dollars.model_healthy_spend/dollars.food_spend
    if not np.allclose(dollars.observed_share, dollars.Y, rtol=1e-9, atol=1e-10):
        raise ValueError("The comparison observed outcome differs from the main spending DID outcome.")
    for column in ["observed_share", "model_implied_share"]:
        values = dollars[column].to_numpy()
        if not np.isfinite(values).all() or ((values < -1e-10) | (values > 1+1e-10)).any():
            raise ValueError(f"Invalid spending-share predictions: {column}.")
    dollars.to_parquet(private_dir/"matched_household_year_healthy_spending_predictions.parquet", index=False)
    category_table.to_parquet(private_dir/"matched_category_healthy_spending_predictions.parquet", index=False)
    pd.DataFrame(parameters).to_csv(table_dir/"healthy_spending_auxiliary_parameters.csv", index=False)
    pd.DataFrame(diagnostics).to_csv(table_dir/"healthy_spending_auxiliary_fit_diagnostics.csv", index=False)
    pd.DataFrame(amount_audits).to_csv(table_dir/"healthy_spending_amount_input_audit.csv", index=False)
    dollars.groupby("panel_year").agg(
        households=("hhid", "nunique"), food_spend=("food_spend", "sum"),
        observed_healthy_spend=("observed_healthy_spend", "sum"), model_healthy_spend=("model_healthy_spend", "sum"),
        mean_household_observed_share=("observed_share", "mean"),
        mean_household_model_implied_share=("model_implied_share", "mean")).reset_index().to_csv(
            table_dir/"healthy_spending_annual_summary.csv", index=False)
    provenance = dict(
        version="V42-fixed-budget-amount-v1", denominator="actual annual sum of food_spend",
        numerator="sum of actual category budgets times predicted conditional healthy fractions",
        model_results_dir=str(model_results_dir), model_files={k: str(v) for k, v in paths.items()},
        state_weights="original smoothed HMM household-year posteriors",
        auxiliary="state-independent, outcome-specific, household-cross-fitted ridge fractional logits",
        auxiliary_covariates=controls, budget_cuts=list(AMOUNT_BUDGET_CUTS),
        auxiliary_ridge=AMOUNT_AUX_RIDGE, auxiliary_folds=AMOUNT_AUX_FOLDS, auxiliary_fold_seed=AMOUNT_AUX_SEED,
        auxiliary_training="positive-budget trips in the HMM estimation household-years; one weight per trip",
        auxiliary_boundary_means="exact zero/one training outcomes use constant zero/one predictions",
        auxiliary_leakage_control="all years of predicted households excluded from their auxiliary training folds",
        choice_marginalization="all source purchase outcomes, including separate low/medium/high cereal gates",
        category_incidence="condition on actual category budget; no extra incidence multiplier",
        matching="main spending-DID pairs, cohorts, reuse and weights held fixed",
        observed_outcome_check="reproduces main spending DID numerator, denominator, and Y",
        n_matched_household_years=len(dollars), confidence_level=CONFIDENCE_LEVEL,
        interpretation="conditional spending allocation fit given observed budgets and smoothed purchase histories",
        uncertainty="observed 90% household-clustered fixed-match intervals; model curve is fitted point predictions",
        model_and_auxiliary_estimation_uncertainty_included=False,
        main_hmm_reestimated=False, main_hmm_likelihood_aic_bic_changed=False,
        method_reference="Papke and Wooldridge (1996), https://www.nber.org/papers/t0147")
    (table_dir.parent/"healthy_spending_comparison_config.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    return dollars, provenance


def plot_healthy_spending_did(results, output_dir):
    """One six-event figure: same spending-share Y and identical matched pairs."""
    import tempfile
    mpl_cache = Path(tempfile.gettempdir()) / "healthy_purchase_matplotlib"
    mpl_cache.mkdir(parents=True, exist_ok=True)
    os.environ["MPLCONFIGDIR"] = str(mpl_cache)
    plot = load_module("matplotlib.pyplot")
    ticker = load_module("matplotlib.ticker")
    lines = load_module("matplotlib.lines")
    plot.rcParams.update({"font.family": "serif", "font.size": 10,
                          "axes.spines.top": False, "axes.spines.right": False,
                          "pdf.fonttype": 42, "ps.fonttype": 42})
    observed = results.loc[results.series.eq("observed")]
    limits = np.r_[observed.ci_low_pp, observed.ci_high_pp, results.did_pp]
    finite = limits[np.isfinite(limits)]
    bound = max(float(np.max(np.abs(finite))) * 1.12 if len(finite) else 0.1, 0.01)
    ticks = ticker.MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10], symmetric=True).tick_values(-bound, bound)
    fig, axes = plot.subplots(2, 3, figsize=(13, 8), sharex=True, sharey=True)
    for ax, (event, label) in zip(axes.flat, EVENT_LABELS.items()):
        sub = results.loc[results.event.eq(event)]
        ax.axhline(0, color="#777777", linewidth=0.8)
        ax.axvline(0, color="#D58B34", linewidth=1.3)
        if len(sub):
            for series, color, marker, style in [("observed", "#245A81", "o", "-"),
                                                  ("model_implied", "#A33B65", "^", "--")]:
                frame = sub.loc[sub.series.eq(series)].sort_values("event_time")
                ax.plot(frame.event_time, frame.did_pp, color=color, linestyle=style,
                        marker=marker, markersize=4, linewidth=1.35, zorder=4)
                if series == "observed":
                    nonref = frame.loc[~frame.is_reference]
                    ax.errorbar(nonref.event_time, nonref.did_pp,
                                yerr=np.vstack([nonref.did_pp-nonref.ci_low_pp,
                                                nonref.ci_high_pp-nonref.did_pp]),
                                fmt="none", ecolor=color, elinewidth=1, capsize=2.5, zorder=3)
            ax.text(0.03, 0.96, f"Treated: {int(sub.matched_treated.iloc[0]):,} | "
                    f"Controls: {int(sub.unique_controls.iloc[0]):,}", transform=ax.transAxes,
                    va="top", fontsize=8, color="#555555")
        else:
            ax.text(0.5, 0.5, "Insufficient eligible matches", ha="center", transform=ax.transAxes)
        ax.set_title(label, pad=10, fontsize=11)
        ax.set_xticks(EVENT_TIMES)
        ax.tick_params(axis="x", labelsize=14)
        ax.set_xlim(-PRE-0.3, POST+0.3)
        ax.set_yticks(ticks)
        ax.set_ylim(ticks[0], ticks[-1])
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:g}"))
        ax.grid(axis="y", color="#E6E6E6", linewidth=0.6)
        ax.set_axisbelow(True)
    handles = [lines.Line2D([], [], color="#245A81", marker="o", label="Observed healthy spending share"),
               lines.Line2D([], [], color="#A33B65", marker="^", linestyle="--", label="HMM + amount allocation")]
    fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.54, 0.995), frameon=False)
    fig.supylabel("DID in healthy food spending share (pp)", x=0.02, fontsize=11)
    fig.supxlabel("Years relative to event (reference: −1)", y=0.12, fontsize=16)
    fig.text(0.085, 0.035,
             "Actual trip-category budgets held fixed; healthy dollars / actual annual food spending.\n"
             "Identical frozen matches and weights. Bars: observed 90% household-clustered CIs conditional on matching.\n"
             "Model curve: HMM choices + cross-fitted amount allocation; HMM/auxiliary estimation uncertainty is not shown.",
             fontsize=8, color="#555555")
    fig.subplots_adjust(left=0.09, right=0.985, top=0.90, bottom=0.22, wspace=0.12, hspace=0.32)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for extension, options in [("png", {"dpi": 300}), ("pdf", {})]:
        fig.savefig(output_dir / f"matched_did_healthy_spending_observed_vs_hmm.{extension}",
                    bbox_inches="tight", **options)
    return fig


def run_healthy_spending_comparison(analysis):
    """Apply the same DID operator twice; never rematch on model predictions."""
    if "matched_pairs" not in analysis:
        raise ValueError("Rerun the main analysis cell to retain its matched pairs in memory.")
    if not any(len(p) for p in analysis["matched_pairs"].values()):
        print("No retained matches; the healthy-spending comparison is unavailable.", flush=True)
        return None
    table_dir, figure_dir = OUTPUT_DIR / "tables", OUTPUT_DIR / "figures"
    dollars, provenance = build_matched_healthy_spending(
        analysis, MODEL_RESULTS_DIR, PRIVATE_DIR, table_dir)
    columns = range(YEAR_START, YEAR_END+1)
    wide = {series: dollars.pivot(index="hhid", columns="panel_year", values=value).reindex(columns=columns)
            for series, value in [("observed", "observed_share"), ("model_implied", "model_implied_share")]}
    estimates, pretests, cohort_results, covariances, sample_audits = [], [], [], [], []
    levels = []
    for event, pairs in analysis["matched_pairs"].items():
        if pairs.empty:
            continue
        pairs = pairs.copy()
        pairs["treated_hhid"] = pairs.treated_hhid.astype(str)
        pairs["control_hhid"] = pairs.control_hhid.astype(str)
        n = len(pairs)
        for series, ywide in wide.items():
            for cohort, group in pairs.groupby("cohort", sort=True):
                years = int(cohort) + EVENT_TIMES
                for arm, id_column in [("treated", "treated_hhid"), ("control", "control_hhid")]:
                    values = ywide.loc[group[id_column], years].to_numpy(dtype=float)
                    if not np.isfinite(values).all():
                        raise ValueError(f"Missing {series} spending share in frozen {event}/{cohort}/{arm} window.")
                    levels.extend(dict(event=event, series=series, cohort=int(cohort), arm=arm,
                                       event_time=int(k), mean_share=float(values[:, j].mean()),
                                       matched_pairs=len(group), cohort_weight=len(group)/n)
                                  for j, k in enumerate(EVENT_TIMES))
            result, pretest, cohorts, covariance = estimate_event(pairs, ywide, event)
            if series == "observed":
                original = analysis["coefficients"].loc[analysis["coefficients"].event.eq(event)].sort_values("event_time")
                compare = ["did_pp", "se_pp", "ci_low_pp", "ci_high_pp"]
                if len(original) != len(result) or not np.allclose(
                        original[compare].to_numpy(), result.sort_values("event_time")[compare].to_numpy(),
                        rtol=1e-8, atol=1e-8, equal_nan=True):
                    raise ValueError("The observed comparison does not reproduce the main spending DID. "
                                     "Rerun the main analysis with the same confidence level and matches.")
            scope = ("fixed-match household-clustered" if series == "observed" else
                     "conditional only: HMM, auxiliary fits and smoothed posteriors held fixed")
            result["series"], result["inference_scope"] = series, scope
            pretest.update(series=series, inference_scope=scope)
            cohorts["series"], covariance["series"] = series, series
            estimates.append(result); pretests.append(pretest)
            cohort_results.append(cohorts); covariances.append(covariance)
        sample_audits.append(dict(event=event, main_did_matched_pairs=n,
                                 observed_spending_matched_pairs=n, model_spending_matched_pairs=n,
                                 main_did_unique_controls=pairs.control_hhid.nunique(),
                                 pairs_dropped_for_model_coverage=0, identical_pair_weights=True))
    results = pd.concat(estimates, ignore_index=True)
    for filename, frame in {
        "healthy_spending_did_observed_vs_hmm.csv": results,
        "healthy_spending_pretrend_tests.csv": pd.DataFrame(pretests),
        "healthy_spending_cohort_did.csv": pd.concat(cohort_results, ignore_index=True),
        "healthy_spending_did_covariance.csv": pd.concat(covariances, ignore_index=True),
        "healthy_spending_matched_levels_by_cohort.csv": pd.DataFrame(levels),
        "healthy_spending_same_matches_audit.csv": pd.DataFrame(sample_audits),
    }.items():
        if any("hhid" in str(c).lower() for c in frame):
            raise RuntimeError(f"Household identifier in aggregate export: {filename}")
        frame.to_csv(table_dir / filename, index=False)
    figure = plot_healthy_spending_did(results, figure_dir)
    display(pd.DataFrame(sample_audits))
    display(figure)
    print(f"Saved healthy-spending comparison: {figure_dir}", flush=True)
    return dict(annual_predictions=dollars, coefficients=results, figure=figure, provenance=provenance)




In [ ]:
if RUN_HEALTHY_SPENDING_COMPARISON:
    healthy_spending_comparison = run_healthy_spending_comparison(analysis)
else:
    print("Healthy-spending comparison disabled in settings.", flush=True)
